# 🚖 Trabajo Práctico: Agentes Reactivos para Refuerzo de Taxis
### Laboratorio de Inteligencia Artificial y Ciencia de Datos — Movilidad Urbana

---

## 📌 Resumen y Mapeo Integral de Archivos

Este notebook **unifica y condensa el 100% del trabajo práctico desarrollado**, estructurado a través de módulos modulares donde **cada pieza de código detalla explícitamente su archivo de origen en el repositorio**.

### 🗺️ Tabla de Correspondencia y Trazabilidad

| Módulo en este Notebook | Archivo de Origen en el Repositorio | Descripción y Rol en el TP |
|---|---|---|
| **1. Importaciones y Entorno** | `requirements.txt` | Dependencias y librerías utilizadas en la solución. |
| **2. Agentes de Movilidad** | [`agentes_movilidad.py`](agentes_movilidad.py) | Lógica del agente reactivo simple, agente basado en modelo, validación de percepciones y procesamiento de secuencias. |
| **3. Simulación y Escenario** | [`simulador_entorno_agente.py`](simulador_entorno_agente.py) <br> [`generar_bitacora.py`](generar_bitacora.py) <br> `escenario_agente/` | Generación del entorno didáctico reproducible, cálculo de cuotas sintéticas ($q_{\text{otras}}$), carga de `percepciones.csv` y creación de `bitacora_agentes.csv`. |
| **4. Pruebas Unitarias** | [`test_agentes_movilidad.py`](test_agentes_movilidad.py) | Batería de 8 tests obligatorios (presión baja, racha, prueba decisiva, percepciones inválidas y causalidad temporal). |
| **5. Informe y Respuestas** | [`informe.md`](informe.md) | Respuestas a las 5 preguntas conceptuales, tabla PEAS completa y reconocimiento de las 6 limitaciones del modelo. |
| **6. Control de Calidad y Rúbrica** | [`CONSIGNAS_DE_REVISION.md`](CONSIGNAS_DE_REVISION.md) <br> [`consigna_agentes_movilidad.md`](consigna_agentes_movilidad.md) | Checklist de verificación de cumplimiento de consignas y matriz de evaluación de rúbrica. |

---

## 1. Importación de Librerías y Configuración

> **Archivos de referencia:** `requirements.txt` y configuración general de dependencias.

In [1]:
from __future__ import annotations

import inspect
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# Configuración de visualización para pandas
pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda x: f"{x:.4f}" if isinstance(x, float) and abs(x) < 1e-2 else (f"{x:.2f}" if isinstance(x, float) else str(x)))

print("✅ Entorno y dependencias cargadas correctamente.")

✅ Entorno y dependencias cargadas correctamente.


---
## 2. Implementación de los Agentes de Movilidad
> **Archivo de origen:** [`agentes_movilidad.py`](agentes_movilidad.py)

En este módulo se definen las estructuras de datos, las validaciones de robustez, el **Agente Reactivo Simple** y el **Agente Reactivo Basado en Modelo**.

### 2.1 Constantes de Dominio y Validación de Percepciones

- **Acciones permitidas:** `NO_REFORZAR`, `RECOMENDAR_REFUERZO`, `ABSTENERSE`.
- **Umbral de presión:** $0.85$ (cociente entre demanda asignada a la empresa X y su capacidad).
- **Validación robusta:** Verificación estricta de campos requeridos (`presion`, `capacidad_x`), exclusión de tipos no numéricos o booleanos, rechazo de infinitos / `NaN` y garantía de que `capacidad_x > 0`.

In [2]:
# ==============================================================================
# ORIGEN: agentes_movilidad.py (Líneas 16-54)
# ==============================================================================

ACCIONES = {"NO_REFORZAR", "RECOMENDAR_REFUERZO", "ABSTENERSE"}
UMBRAL_PRESION = 0.85

# Campos que deben estar presentes y ser numericos validos para poder decidir.
CAMPOS_REQUERIDOS = ("presion", "capacidad_x")


def _es_numero_valido(valor: Any) -> bool:
    """Comprueba si un valor es un numero real finito y no booleano."""
    if valor is None:
        return False
    if isinstance(valor, bool):
        return False
    try:
        numero = float(valor)
    except (TypeError, ValueError):
        return False
    return math.isfinite(numero)


def _percepcion_valida(percepcion: dict[str, Any]) -> bool:
    """Valida que la percepcion tenga los campos requeridos y sean usables.

    Se considera invalida cuando falta un campo, cuando el valor no es un
    numero finito (incluye NaN e infinitos, que surgen cuando la capacidad
    es cero y por lo tanto la presion queda indefinida) o cuando la
    capacidad reportada no es positiva, es decir, la capacidad de X es
    desconocida o nula.
    """
    for campo in CAMPOS_REQUERIDOS:
        if campo not in percepcion or not _es_numero_valido(percepcion[campo]):
            return False

    if float(percepcion["capacidad_x"]) <= 0:
        return False
    if float(percepcion["presion"]) < 0:
        return False

    return True

### 2.2 Parte 1: Agente Reactivo Simple
> **Archivo de origen:** [`agentes_movilidad.py`](agentes_movilidad.py) (función `decidir_reactivo_simple`)

El agente reactivo simple aplica una regla directa condición-acción sobre la percepción del instante actual:
1. Si los datos son inválidos o la capacidad es desconocida $\rightarrow$ `ABSTENERSE`.
2. Si $\text{presion} \ge 0.85$ $\rightarrow$ `RECOMENDAR_REFUERZO`.
3. Si $\text{presion} < 0.85$ $\rightarrow$ `NO_REFORZAR`.

*Nota:* No utiliza variables globales mutables ni historial previo ni datos del futuro.

In [3]:
# ==============================================================================
# ORIGEN: agentes_movilidad.py (Líneas 56-74)
# ==============================================================================

def decidir_reactivo_simple(percepcion: dict[str, Any]) -> tuple[str, str]:
    """Devuelve (accion, motivo) usando solo la percepcion actual."""
    if not _percepcion_valida(percepcion):
        return (
            "ABSTENERSE",
            "Percepcion invalida, incompleta o con capacidad desconocida.",
        )

    presion = float(percepcion["presion"])
    if presion >= UMBRAL_PRESION:
        return (
            "RECOMENDAR_REFUERZO",
            f"Presion actual {presion:.2f} >= umbral {UMBRAL_PRESION:.2f}.",
        )
    return (
        "NO_REFORZAR",
        f"Presion actual {presion:.2f} < umbral {UMBRAL_PRESION:.2f}.",
    )

### 2.3 Parte 2: Agente Reactivo Basado en Modelo
> **Archivo de origen:** [`agentes_movilidad.py`](agentes_movilidad.py) (funciones `crear_estado_inicial`, `actualizar_estado`, `decidir_reactivo_modelo`)

El agente basado en modelo mantiene un **estado interno** que resume aspectos no directamente observables en una única medición (la persistencia temporal de la demanda alta):
- **Estado mínimo:** `percepcion_valida`, `racha_presion_alta`, `presion_anterior`, `ultima_accion`.
- **Actualización:** Incrementa `racha_presion_alta` en 1 si $\text{presion} \ge 0.85$; la reinicia a 0 si es menor.
- **Regla de decisión:**
  1. Si `percepcion_valida` es `False` $\rightarrow$ `ABSTENERSE`.
  2. Si `racha_presion_alta >= 2` $\rightarrow$ `RECOMENDAR_REFUERZO` (filtro contra picos aislados de ruido).
  3. Cualquier otro estado válido $\rightarrow$ `NO_REFORZAR`.

In [4]:
# ==============================================================================
# ORIGEN: agentes_movilidad.py (Líneas 76-131)
# ==============================================================================

def crear_estado_inicial() -> dict[str, Any]:
    """Crea el estado persistente del agente reactivo basado en modelo."""
    return {
        "percepcion_valida": False,
        "racha_presion_alta": 0,
        "presion_anterior": None,
        "ultima_accion": None,
    }


def actualizar_estado(
    estado_anterior: dict[str, Any],
    percepcion: dict[str, Any],
) -> dict[str, Any]:
    """Actualiza la memoria a partir del estado anterior y la percepcion."""
    if not _percepcion_valida(percepcion):
        return {
            "percepcion_valida": False,
            "racha_presion_alta": 0,
            "presion_anterior": None,
            "ultima_accion": estado_anterior.get("ultima_accion"),
        }

    presion = float(percepcion["presion"])
    racha_anterior = estado_anterior.get("racha_presion_alta", 0) or 0
    racha_actual = racha_anterior + 1 if presion >= UMBRAL_PRESION else 0

    return {
        "percepcion_valida": True,
        "racha_presion_alta": racha_actual,
        "presion_anterior": presion,
        "ultima_accion": estado_anterior.get("ultima_accion"),
    }


def decidir_reactivo_modelo(
    estado_actual: dict[str, Any],
) -> tuple[str, str]:
    """Devuelve (accion, motivo) a partir del estado interno actualizado."""
    if not estado_actual.get("percepcion_valida", False):
        return (
            "ABSTENERSE",
            "El estado actual proviene de una percepcion invalida.",
        )

    racha = estado_actual.get("racha_presion_alta", 0)
    if racha >= 2:
        return (
            "RECOMENDAR_REFUERZO",
            f"Presion alta sostenida durante {racha} horas consecutivas.",
        )
    return (
        "NO_REFORZAR",
        f"Racha de presion alta de {racha} hora(s); no alcanza la persistencia minima.",
    )

### 2.4 Parte 3: Procesamiento de Secuencias Temporales
> **Archivo de origen:** [`agentes_movilidad.py`](agentes_movilidad.py) (función `procesar_secuencia`)

Ejecuta ambos agentes a lo largo de un flujo cronológico de observaciones para generar la bitácora comparativa.

In [5]:
# ==============================================================================
# ORIGEN: agentes_movilidad.py (Líneas 133-170)
# ==============================================================================

def procesar_secuencia(percepciones: pd.DataFrame) -> pd.DataFrame:
    """Ejecuta ambos agentes y construye la bitacora comparativa."""
    filas: list[dict[str, Any]] = []
    estado = crear_estado_inicial()

    for _, fila in percepciones.sort_values("hora").iterrows():
        percepcion = fila.to_dict()

        accion_simple, motivo_simple = decidir_reactivo_simple(percepcion)
        estado = actualizar_estado(estado, percepcion)
        accion_modelo, motivo_modelo = decidir_reactivo_modelo(estado)
        estado["ultima_accion"] = accion_modelo

        filas.append(
            {
                "hora": int(percepcion.get("hora")),
                "presion": float(percepcion.get("presion")),
                "racha_presion_alta": int(estado["racha_presion_alta"]),
                "accion_simple": accion_simple,
                "motivo_simple": motivo_simple,
                "accion_modelo": accion_modelo,
                "motivo_modelo": motivo_modelo,
            }
        )

    return pd.DataFrame(
        filas,
        columns=[
            "hora",
            "presion",
            "racha_presion_alta",
            "accion_simple",
            "motivo_simple",
            "accion_modelo",
            "motivo_modelo",
        ],
    )

---
## 3. Simulación del Entorno, Escenario Reproducible y Bitácora
> **Archivos de origen:** 
> - [`simulador_entorno_agente.py`](simulador_entorno_agente.py) (lógica de simulación sintética de cuotas de mercado)
> - [`generar_bitacora.py`](generar_bitacora.py) (construcción y exportación de bitácora)
> - `escenario_agente/percepciones.csv` (dataset de percepción hasta hora $h$)
> - `escenario_agente/resultado_h_mas_1.csv` (evaluación futura de control para hora $h+1$)
> - `bitacora_agentes.csv` (archivo de bitácora resultante de la ejecución)

### 3.1 Fundamentos de la Simulación Sintética del Entorno
En el simulador, la cuota de mercado de otras empresas $q_{\text{otras}}$ se modela sintéticamente con la fórmula:

$$
q_{\text{otras}} = \operatorname{clip}\left( q_{\min} + \frac{q_{\max} - q_{\min}}{1 + n_X / n_{\text{ref}}} + \varepsilon,\; q_{\min},\; q_{\max} \right), \quad \varepsilon \sim \mathcal{N}(0, \sigma)
$$

Donde $q_{\min} = 0.15$, $q_{\max} = 0.75$, $n_{\text{ref}} = 10.0$, $\sigma = 0.05$.

In [6]:
# ==============================================================================
# ORIGEN: simulador_entorno_agente.py (Líneas 34-92)
# ==============================================================================

Q_OTRAS_MIN = 0.15
Q_OTRAS_MAX = 0.75
TAXIS_REFERENCIA = 10.0
SIGMA_OTRAS = 0.05


def tasa_otras_esperada(
    taxis_x: int,
    q_min: float = Q_OTRAS_MIN,
    q_max: float = Q_OTRAS_MAX,
    taxis_referencia: float = TAXIS_REFERENCIA,
) -> float:
    """Calcula la cuota base de otras empresas, inversa a la flota de X."""
    if taxis_x < 0:
        raise ValueError("taxis_x no puede ser negativo")
    return q_min + (q_max - q_min) / (1 + taxis_x / taxis_referencia)


def sortear_tasa_otras(
    taxis_x: int,
    generador: np.random.Generator,
    q_min: float = Q_OTRAS_MIN,
    q_max: float = Q_OTRAS_MAX,
    taxis_referencia: float = TAXIS_REFERENCIA,
    sigma: float = SIGMA_OTRAS,
) -> float:
    """Agrega ruido normal a la cuota base y la mantiene entre sus limites."""
    base = tasa_otras_esperada(taxis_x, q_min, q_max, taxis_referencia)
    return float(np.clip(base + generador.normal(0, sigma), q_min, q_max))


# Demostración del comportamiento de la tasa según flota de X
flotas = [5, 10, 20, 40]
rng_demo = np.random.default_rng(42)
print("--- Demostración de q_otras según tamaño de flota de X ---")
for f in flotas:
    t_esp = tasa_otras_esperada(f)
    t_sort = sortear_tasa_otras(f, rng_demo)
    print(f"Flota X: {f:2d} taxis | Tasa esperada: {t_esp:.4f} | Tasa sorteada: {t_sort:.4f}")

--- Demostración de q_otras según tamaño de flota de X ---
Flota X:  5 taxis | Tasa esperada: 0.5500 | Tasa sorteada: 0.5652
Flota X: 10 taxis | Tasa esperada: 0.4500 | Tasa sorteada: 0.3980
Flota X: 20 taxis | Tasa esperada: 0.3500 | Tasa sorteada: 0.3875
Flota X: 40 taxis | Tasa esperada: 0.2700 | Tasa sorteada: 0.3170


### 3.2 Carga del Escenario Reproducible Oficial
> **Archivo de origen:** `escenario_agente/percepciones.csv`

Parámetros del escenario reproducible según consigna:
- **Zona TLC:** `161` (Midtown Center)
- **Hora de corte:** `8` (horas visibles: 6, 7 y 8)
- **Flota de X:** `20` taxis
- **Horas de historia:** `3`
- **Semilla aleatoria:** `42`

In [7]:
# ==============================================================================
# ORIGEN: escenario_agente/percepciones.csv & generar_bitacora.py
# ==============================================================================

ruta_percepciones = Path("escenario_agente/percepciones.csv")

if ruta_percepciones.exists():
    df_percepciones = pd.read_csv(ruta_percepciones)
    print("📁 Archivo 'escenario_agente/percepciones.csv' cargado exitosamente:")
    display(df_percepciones)
else:
    print("⚠️ El archivo local no se encontró. Creando dataframe con los datos reproducibles exactos:")
    df_percepciones = pd.DataFrame([
        {"zona_id": 161, "zona": "Midtown Center", "hora": 6, "taxis_x": 20, "demanda_total": 37, "tasa_otras_simulada": 0.2980007946879752, "viajes_otras": 14, "demanda_x": 23, "capacidad_x": 20, "viajes_atendibles_x": 20, "demanda_no_cubierta_x": 3, "presion": 1.15},
        {"zona_id": 161, "zona": "Midtown Center", "hora": 7, "taxis_x": 20, "demanda_total": 61, "tasa_otras_simulada": 0.25244824056730814, "viajes_otras": 22, "demanda_x": 39, "capacidad_x": 20, "viajes_atendibles_x": 20, "demanda_no_cubierta_x": 19, "presion": 1.95},
        {"zona_id": 161, "zona": "Midtown Center", "hora": 8, "taxis_x": 20, "demanda_total": 91, "tasa_otras_simulada": 0.3341878703828209, "viajes_otras": 29, "demanda_x": 62, "capacidad_x": 20, "viajes_atendibles_x": 20, "demanda_no_cubierta_x": 42, "presion": 3.10}
    ])
    display(df_percepciones)

📁 Archivo 'escenario_agente/percepciones.csv' cargado exitosamente:


,zona_id,zona,hora,taxis_x,demanda_total,tasa_otras_simulada,viajes_otras,demanda_x,capacidad_x,viajes_atendibles_x,demanda_no_cubierta_x,presion
0,161,Midtown Center,6,20,37,0.30,14,23,20,20,3,1.15
1,161,Midtown Center,7,20,61,0.25,22,39,20,20,19,1.95
2,161,Midtown Center,8,20,91,0.33,29,62,20,20,42,3.10


### 3.3 Generación y Visualización de la Bitácora Comparativa
> **Archivos de origen:** [`generar_bitacora.py`](generar_bitacora.py) y `bitacora_agentes.csv`

In [8]:
# ==============================================================================
# ORIGEN: generar_bitacora.py (Líneas 20-32) & bitacora_agentes.csv
# ==============================================================================

bitacora_generada = procesar_secuencia(df_percepciones)

print("📋 Bitácora Comparativa de Decisiones (bitacora_agentes.csv):")
display(bitacora_generada)

# Verificación de concordancia exacta con bitacora_agentes.csv en disco si existe
ruta_bitacora_disco = Path("bitacora_agentes.csv")
if ruta_bitacora_disco.exists():
    df_disco = pd.read_csv(ruta_bitacora_disco)
    pd.testing.assert_frame_equal(bitacora_generada, df_disco)
    print("✅ La bitácora coincide exactamente con 'bitacora_agentes.csv'.")

📋 Bitácora Comparativa de Decisiones (bitacora_agentes.csv):


,hora,presion,racha_presion_alta,accion_simple,motivo_simple,accion_modelo,motivo_modelo
0,6,1.15,1,RECOMENDAR_REFUERZO,Presion actual 1.15 >= umbral 0.85.,NO_REFORZAR,Racha de presion alta de 1 hora(s); no alcanza...
1,7,1.95,2,RECOMENDAR_REFUERZO,Presion actual 1.95 >= umbral 0.85.,RECOMENDAR_REFUERZO,Presion alta sostenida durante 2 horas consecu...
2,8,3.10,3,RECOMENDAR_REFUERZO,Presion actual 3.10 >= umbral 0.85.,RECOMENDAR_REFUERZO,Presion alta sostenida durante 3 horas consecu...


✅ La bitácora coincide exactamente con 'bitacora_agentes.csv'.


### 3.4 Inspección de la Evaluación Futura de Control ($h+1$)
> **Archivo de origen:** `escenario_agente/resultado_h_mas_1.csv`

> ⚠️ **Principio de Causalidad Temporal:**
> Este archivo contiene la hora 9 ($h+1$) y está reservado exclusivamente para análisis o evaluación posterior humana. **Nunca forma parte de la percepción de los agentes** en el instante $h=8$.

In [9]:
# ==============================================================================
# ORIGEN: escenario_agente/resultado_h_mas_1.csv (Evaluación posterior de control)
# ==============================================================================

ruta_futuro = Path("escenario_agente/resultado_h_mas_1.csv")
if ruta_futuro.exists():
    df_futuro = pd.read_csv(ruta_futuro)
    print("📊 Datos de control posterior para h+1 (Hora 9):")
    display(df_futuro)
else:
    print("ℹ️ Archivo 'resultado_h_mas_1.csv' reservado.")

📊 Datos de control posterior para h+1 (Hora 9):


,zona_id,zona,hora,taxis_x,demanda_total,tasa_otras_simulada,viajes_otras,demanda_x,capacidad_x,viajes_atendibles_x,demanda_no_cubierta_x,presion,necesita_refuerzo,taxis_adicionales_sugeridos
0,161,Midtown Center,9,20,125,0.39,56,69,20,20,49,3.45,True,49


---
## 4. Batería de Pruebas Unitarias Obligatorias
> **Archivo de origen:** [`test_agentes_movilidad.py`](test_agentes_movilidad.py)

En este módulo se ejecutan y validan los 8 tests unitarios exigidos por la consigna para garantizar el correcto funcionamiento de todas las reglas lógicas, la memoria de estado y la causalidad temporal.

In [10]:
# ==============================================================================
# ORIGEN: test_agentes_movilidad.py (Líneas 23-48)
# Helper para construir percepciones sintéticas en los tests
# ==============================================================================

def _percepcion(
    hora: int,
    presion: float,
    capacidad_x: float = 10,
    demanda_x: float | None = None,
    taxis_x: int = 10,
) -> dict[str, Any]:
    """Arma una percepcion de ejemplo con la misma forma que produce el simulador."""
    if demanda_x is None:
        demanda_x = round(presion * capacidad_x)
    return {
        "zona_id": 161,
        "zona": "Zona de prueba",
        "hora": hora,
        "taxis_x": taxis_x,
        "demanda_total": demanda_x + 5,
        "tasa_otras_simulada": 0.3,
        "viajes_otras": 5,
        "demanda_x": demanda_x,
        "capacidad_x": capacidad_x,
        "viajes_atendibles_x": min(demanda_x, capacidad_x),
        "demanda_no_cubierta_x": max(demanda_x - capacidad_x, 0),
        "presion": presion,
    }

### 4.1 Ejecución Interactiva de cada Caso de Test

A continuación definimos y ejecutamos cada una de las pruebas que conforman el archivo `test_agentes_movilidad.py`:

In [11]:
# ==============================================================================
# ORIGEN: test_agentes_movilidad.py (Líneas 50-177)
# ==============================================================================

def test_presion_baja_ambos_no_refuerzan():
    percepcion = _percepcion(hora=8, presion=0.5)
    accion_simple, _ = decidir_reactivo_simple(percepcion)
    estado = actualizar_estado(crear_estado_inicial(), percepcion)
    accion_modelo, _ = decidir_reactivo_modelo(estado)

    assert accion_simple == "NO_REFORZAR", f"Esperado NO_REFORZAR, obtenido {accion_simple}"
    assert accion_modelo == "NO_REFORZAR", f"Esperado NO_REFORZAR, obtenido {accion_modelo}"
    print("  ✓ Test 1 PASSED: Presión baja -> Ambos NO_REFORZAR.")


def test_primera_hora_presion_alta_solo_simple_recomienda():
    percepcion = _percepcion(hora=8, presion=0.9)
    accion_simple, _ = decidir_reactivo_simple(percepcion)
    estado = actualizar_estado(crear_estado_inicial(), percepcion)
    accion_modelo, _ = decidir_reactivo_modelo(estado)

    assert accion_simple == "RECOMENDAR_REFUERZO"
    assert accion_modelo == "NO_REFORZAR"
    assert estado["racha_presion_alta"] == 1
    print("  ✓ Test 2 PASSED: Primera hora alta -> Simple RECOMIENDA, Modelo NO.")


def test_segunda_hora_consecutiva_ambos_recomiendan():
    estado = crear_estado_inicial()
    estado = actualizar_estado(estado, _percepcion(hora=8, presion=0.9))

    percepcion_2 = _percepcion(hora=9, presion=0.95)
    accion_simple, _ = decidir_reactivo_simple(percepcion_2)
    estado = actualizar_estado(estado, percepcion_2)
    accion_modelo, _ = decidir_reactivo_modelo(estado)

    assert accion_simple == "RECOMENDAR_REFUERZO"
    assert accion_modelo == "RECOMENDAR_REFUERZO"
    assert estado["racha_presion_alta"] == 2
    print("  ✓ Test 3 PASSED: Segunda hora consecutiva -> Ambos RECOMIENDAN.")


def test_misma_percepcion_final_con_historias_distintas():
    """PRUEBA DECISIVA: Demuestra que el agente con modelo usa el historial."""
    percepcion_final = _percepcion(hora=8, presion=0.9)

    # Historia A: hora 7 con presión baja, hora 8 con presión alta (racha=1)
    estado_a = actualizar_estado(crear_estado_inicial(), _percepcion(hora=7, presion=0.2))
    estado_a = actualizar_estado(estado_a, percepcion_final)

    # Historia B: hora 7 YA con presión alta, hora 8 repite (racha=2)
    estado_b = actualizar_estado(crear_estado_inicial(), _percepcion(hora=7, presion=0.9))
    estado_b = actualizar_estado(estado_b, percepcion_final)

    accion_simple_a, _ = decidir_reactivo_simple(percepcion_final)
    accion_simple_b, _ = decidir_reactivo_simple(percepcion_final)
    accion_modelo_a, _ = decidir_reactivo_modelo(estado_a)
    accion_modelo_b, _ = decidir_reactivo_modelo(estado_b)

    # El agente simple no tiene memoria: misma percepción -> misma acción
    assert accion_simple_a == accion_simple_b == "RECOMENDAR_REFUERZO"

    # El agente con memoria sí distingue el historial previo
    assert accion_modelo_a == "NO_REFORZAR"
    assert accion_modelo_b == "RECOMENDAR_REFUERZO"
    assert accion_modelo_a != accion_modelo_b
    print("  ✓ Test 4 PASSED (Prueba Decisiva): Misma percepción final con historias distintas.")


def test_capacidad_desconocida_abstiene():
    # capacidad_x=0 hace que la presión quede indefinida (infinita)
    percepcion = _percepcion(hora=8, presion=float("inf"), capacidad_x=0, demanda_x=5)
    accion_simple, _ = decidir_reactivo_simple(percepcion)
    estado = actualizar_estado(crear_estado_inicial(), percepcion)
    accion_modelo, _ = decidir_reactivo_modelo(estado)

    assert accion_simple == "ABSTENERSE"
    assert accion_modelo == "ABSTENERSE"
    print("  ✓ Test 5 PASSED: Capacidad desconocida/nula -> Ambos ABSTENERSE.")


def test_campo_faltante_abstiene():
    percepcion = _percepcion(hora=8, presion=0.9)
    del percepcion["capacidad_x"]
    accion_simple, _ = decidir_reactivo_simple(percepcion)

    assert accion_simple == "ABSTENERSE"
    print("  ✓ Test 6 PASSED: Campo faltante -> ABSTENERSE.")


def test_funciones_de_decision_no_reciben_datos_futuros():
    assert list(inspect.signature(decidir_reactivo_simple).parameters) == ["percepcion"]
    assert list(inspect.signature(decidir_reactivo_modelo).parameters) == ["estado_actual"]
    print("  ✓ Test 7 PASSED: Firmas de función no aceptan parámetros de h+1.")


def test_procesar_secuencia_ignora_cambios_futuros():
    percepciones = pd.DataFrame([
        _percepcion(hora=7, presion=0.3),
        _percepcion(hora=8, presion=0.9),
        _percepcion(hora=9, presion=0.9),
    ])
    bitacora_original = procesar_secuencia(percepciones)

    # Alteramos solo la última hora (h=9, la más "futura")
    percepciones_alteradas = percepciones.copy()
    percepciones_alteradas.loc[percepciones_alteradas["hora"] == 9, "presion"] = 0.0
    bitacora_alterada = procesar_secuencia(percepciones_alteradas)

    pasado_original = bitacora_original.loc[bitacora_original["hora"] < 9].reset_index(drop=True)
    pasado_alterado = bitacora_alterada.loc[bitacora_alterada["hora"] < 9].reset_index(drop=True)

    pd.testing.assert_frame_equal(pasado_original, pasado_alterado)
    print("  ✓ Test 8 PASSED: Causalidad temporal (alteraciones futuras no afectan decisiones pasadas).")

### 4.2 Ejecución de la Suite Completa de Pruebas

In [12]:
print("🧪 Ejecutando la suite de 8 tests obligatorios:")
print("=" * 60)

test_presion_baja_ambos_no_refuerzan()
test_primera_hora_presion_alta_solo_simple_recomienda()
test_segunda_hora_consecutiva_ambos_recomiendan()
test_misma_percepcion_final_con_historias_distintas()
test_capacidad_desconocida_abstiene()
test_campo_faltante_abstiene()
test_funciones_de_decision_no_reciben_datos_futuros()
test_procesar_secuencia_ignora_cambios_futuros()

print("=" * 60)
print("🎉 ¡TODOS LOS TESTS PASARON EXITOSAMENTE (8/8)! ✨")

🧪 Ejecutando la suite de 8 tests obligatorios:
  ✓ Test 1 PASSED: Presión baja -> Ambos NO_REFORZAR.
  ✓ Test 2 PASSED: Primera hora alta -> Simple RECOMIENDA, Modelo NO.
  ✓ Test 3 PASSED: Segunda hora consecutiva -> Ambos RECOMIENDAN.
  ✓ Test 4 PASSED (Prueba Decisiva): Misma percepción final con historias distintas.
  ✓ Test 5 PASSED: Capacidad desconocida/nula -> Ambos ABSTENERSE.
  ✓ Test 6 PASSED: Campo faltante -> ABSTENERSE.
  ✓ Test 7 PASSED: Firmas de función no aceptan parámetros de h+1.
  ✓ Test 8 PASSED: Causalidad temporal (alteraciones futuras no afectan decisiones pasadas).
🎉 ¡TODOS LOS TESTS PASARON EXITOSAMENTE (8/8)! ✨


---
## 5. Informe Técnico, PEAS y Respuestas Teórico-Prácticas
> **Archivo de origen:** [`informe.md`](informe.md)

### 5.1 Respuestas a las Preguntas de la Consigna

#### 1. ¿En qué situaciones ambos agentes producen la misma acción?
Coinciden en dos situaciones principales:
1. **Presión baja:** Cuando $\text{presion} < 0.85$, ninguno de los dos encuentra justificación para recomendar más vehículos (`NO_REFORZAR`).
2. **Presión alta sostenida:** Cuando la presión alta se mantiene durante **dos o más horas consecutivas** (racha $\ge 2$), ambos cumplen sus respectivas condiciones y devuelven `RECOMENDAR_REFUERZO`. En la bitácora reproducible esto se verifica en las horas **7 y 8**.
3. **Percepción inválida o incompleta:** En caso de datos faltantes, `NaN`, infinitos o capacidad $\le 0$, ambos se abstienen (`ABSTENERSE`).

#### 2. ¿Cuándo reaccionan de forma diferente?
Difieren en los **momentos de transición**: en la primera hora en que la presión supera el umbral ($0.85$) tras un período bajo o al inicio de la observación.
- El **agente reactivo simple** no posee memoria y reacciona de manera inmediata a la observación aislada emitiendo `RECOMENDAR_REFUERZO`.
- El **agente basado en modelo** requiere confirmar la persistencia del fenómeno durante al menos dos horas consecutivas (filtro de ruido), por lo que en la primera hora mantiene `NO_REFORZAR` (racha = 1).
- En la bitácora reproducible esto ocurre en la **hora 6** (presión 1.15, racha 1: Simple recomienda, Modelo no).

#### 3. ¿Por qué el segundo agente está basado en modelo aunque no planifique?
Un agente "basado en modelo" no necesita realizar búsqueda de caminos, simulación de estados futuros o planificación de secuencias de acciones hacia metas. Se clasifica como tal porque **mantiene un modelo interno del mundo** que rastrea aspectos del entorno no observables directamente en la percepción instantánea (en este caso, cuántas horas consecutivas lleva la presión alta). La decisión sigue siendo una regla condición-acción (es reactivo), pero la condición se evalúa sobre el **estado interno actualizado** en lugar de los datos sensoriales crudos.

#### 4. ¿Qué representa `tasa_otras_simulada` y qué no permite afirmar?
Representa una proporción sintética de viajes asignados a competidores ficticios en el entorno de simulación, calculada mediante una curva didáctica inversa a la flota de X más ruido gaussiano. 
- **Qué no permite afirmar:** No permite inferir la participación real de mercado de ninguna empresa en Nueva York, ni asumir que existe una relación causal real entre la cantidad de taxis de una empresa y la demanda absorbida por sus competidores.

#### 5. ¿Por qué `resultado_h_mas_1.csv` no puede formar parte de la percepción?
Porque contiene información del estado del sistema en la hora $h+1$, un instante temporal que todavía no ha sucedido al momento de tomar la decisión en la hora $h$. Incorporarlo en la percepción representaría una **fuga de información temporal (data leakage)** y violaría la causalidad estricta requerida por cualquier agente inteligente que opere en tiempo real.

---

### 5.2 Estructuración PEAS del Entorno de Movilidad

| Elemento | Descripción Contextualizada en el Escenario del TP |
|---|---|
| **Performance (Rendimiento)** | Recomendaciones coherentes con los umbrales de presión y persistencia de racha, ausencia total de fuga temporal ($h+1$), abstención ante datos erróneos o incompletos, y justificación trazable en cada decisión (`motivo_simple` / `motivo_modelo`). |
| **Environment (Entorno)** | Secuencia zona-hora simulada (Zona TLC 161 - Midtown Center, horas 6 a 8), demanda histórica TLC Yellow Taxi transformada sintéticamente, flota fija de 20 taxis de la empresa X, competidores simulados y un tomador de decisión humano. |
| **Actuators (Actuadores)** | Emisión de los mensajes discretos: `NO_REFORZAR`, `RECOMENDAR_REFUERZO` y `ABSTENERSE`. El agente no ejecuta traslados físicos ni asignaciones automáticas de vehículos. |
| **Sensors (Sensors)** | Lectura lógica estructurada de `percepciones.csv` fila por fila en orden cronológico estricto (no es un sensor telemático en tiempo real). |

---

### 5.3 Limitaciones y Supuestos Reconocidos

1. **Naturaleza de los datos TLC:** Los registros representan viajes de Yellow Taxi efectivamente realizados y reportados, no la demanda latente total de movilidad ni solicitudes desestimadas.
2. **Empresas ficticias:** La empresa X y sus competidoras son abstracciones creadas con propósitos didácticos.
3. **Hipótesis de cuota sintética:** La relación inversa entre la flota de X y la cuota externa (`tasa_otras_simulada`) es una fórmula teórica, no un modelo econométrico estimado sobre datos reales.
4. **Capacidad unitaria simplificada:** Se asume que cada taxi atiende exactamente 1 viaje por hora, obviando tiempos de viaje, congestión, reposicionamiento y disponibilidad operativa.
5. **Geometría y distancias:** Las distancias euclidianas entre centroides de zona no determinan duraciones de viaje ni disponibilidad vehicular real.
6. **Alcance de la recomendación:** `RECOMENDAR_REFUERZO` es una sugerencia informativa para evaluación humana, no una orden de despacho ni un despacho automático.
7. **Efecto de borde del historial:** Al iniciar la observación en la hora 6, el estado interno se inicializa en 0 (`crear_estado_inicial`). Si la presión venía alta antes de la ventana observable, el agente con memoria no puede saberlo y trata la primera hora observada como inicio de racha.

---
## 6. Verificación de Rúbrica y Consignas de Revisión
> **Archivos de origen:** [`CONSIGNAS_DE_REVISION.md`](CONSIGNAS_DE_REVISION.md) y [`consigna_agentes_movilidad.md`](consigna_agentes_movilidad.md)

### 📊 Matriz de Evaluación Cruzada (10 / 10 Puntos)

| Criterio de Evaluación (2 pts c/u) | Evidencia en el Código y Archivo de Origen | Estado de Verificación |
|---|---|:---:|
| **1. Agente reactivo simple y validación** | [`agentes_movilidad.py`](agentes_movilidad.py): `decidir_reactivo_simple` + `_percepcion_valida` que valida `presion`, `capacidad_x > 0`, descarta `NaN`/infinitos y booleanos. Tests 5 y 6. | ✅ Cumplido |
| **2. Actualización correcta del estado** | [`agentes_movilidad.py`](agentes_movilidad.py): `actualizar_estado` + `crear_estado_inicial` con las 4 claves requeridas. Racha incrementa si $\ge 0.85$ y reinicia en 0 si $< 0.85$. Tests 2 y 3. | ✅ Cumplido |
| **3. Política del agente basado en modelo** | [`agentes_movilidad.py`](agentes_movilidad.py): `decidir_reactivo_modelo` que exige $\text{racha} \ge 2$ para recomendar refuerzo y se abstiene si el estado proviene de percepción inválida. | ✅ Cumplido |
| **4. Pruebas, bitácora y memoria histórica** | [`test_agentes_movilidad.py`](test_agentes_movilidad.py): 8 tests aprobados, `bitacora_agentes.csv` reproducible y Test 4 (prueba decisiva de historias distintas). | ✅ Cumplido |
| **5. PEAS, causalidad y limitaciones** | [`informe.md`](informe.md): PEAS completo contextualizado, respuestas a las 5 preguntas, 6 limitaciones reconocidas y Tests 7-8 de causalidad temporal. | ✅ Cumplido |

---

### 🎓 Guía para la Defensa Individual

- **Diferencia entre agentes:** El simple reacciona al instante presente $h$ (sin memoria); el basado en modelo almacena un resumen de la historia (racha) para confirmar persistencia y filtrar anomalías o picos esporádicos.
- **Garantía de causalidad:** Las funciones de decisión solo reciben la percepción o estado actual (`inspect.signature` lo verifica). En ningún punto se importa ni consulta `resultado_h_mas_1.csv`.
- **Naturaleza de $q_{\text{otras}}$:** Es un artificio didáctico sintético con ruido para dividir la demanda agregada de TLC, no un parámetro empírico de mercado.
- **Sensibilidad de la racha mínima:**
  - Si $\text{racha} = 1$: El agente basado en modelo degeneraría exactamente en el agente reactivo simple.
  - Si $\text{racha} = 3$: Exigiría 3 horas consecutivas de presión alta, aumentando la robustez contra falsos positivos pero introduciendo mayor retraso (inercia) ante incrementos reales y sostenidos de demanda.